In [ ]:
# ! pip install -q evaluate bitsandbytes
# ! pip install -qU datasets wandb

In [ ]:
%%writefile main.py

import os, math, torch, evaluate, wandb, tempfile
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader
from kaggle_secrets import UserSecretsClient

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    get_scheduler,
    AutoModelForSequenceClassification,
)

from accelerate import Accelerator
from accelerate.utils import set_seed

# Config
checkpoint = "bert-base-uncased"
num_train_epochs = 2
train_bs = 16
gacc_steps = 2
lr_scheduler_type = "linear"
lr = 5e-5
max_grad_norm = 1.0
use_8bit_adam = True
logging_steps = 10
eval_on_start = True

os.environ["WANDB_LOG_MODEL"] = "end"
os.environ["WANDB_WATCH"] = "false"
wandb_project = "HFLLM-transformer-finetuning"
wandb_group   = "[accelerate]bert-mrpc-analysis"
wandb_run_name = "test_run_2"

# W&B Login
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")


# Distributed training entrypoint (Accelerate)
def main():
    """
    This function is executed ON EACH PROCESS (GPU).
    With 2 GPUs, it runs twice in parallel.
    """

    # -----------------------------
    # 0) Accelerator setup
    # -----------------------------
    # Detect bf16 support (T4 generally does NOT support bf16 efficiently; fp16 is typical)
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"

    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=mixed_precision,
        log_with="wandb",
    )

    # optional but recommended: reproducibility across processes
    set_seed(42)

    # Only the main process should print a lot
    if accelerator.is_main_process:
        print("Distributed type:", accelerator.state.distributed_type)
        print("Num processes:", accelerator.num_processes)
        print("Mixed precision:", accelerator.state.mixed_precision)

    # -----------------------------
    # 1) W&B tracking (Accelerate-managed)
    # -----------------------------
    # init_trackers initializes wandb run ONLY on the main process
    accelerator.init_trackers(
        project_name=wandb_project,
        config={
            "checkpoint": checkpoint,
            "num_train_epochs": num_train_epochs,
            "train_bs": train_bs,
            "gradient_accumulation_steps": gacc_steps,
            "effective_batch_size": train_bs * gacc_steps * accelerator.num_processes,
            "lr": lr,
            "lr_scheduler_type": lr_scheduler_type,
            "max_grad_norm": max_grad_norm,
            "use_8bit_adam": use_8bit_adam,
            "mixed_precision": mixed_precision,
            "num_gpus": accelerator.num_processes,
        },
        init_kwargs={
            "wandb": {"group": wandb_group, "name": wandb_run_name}
        },
    )

    # -----------------------------
    # 2) Load + tokenize dataset
    # -----------------------------
    raw_datasets = load_dataset("glue", "mrpc")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def tokenize_function(example):
        return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

    with accelerator.main_process_first():
        # The main process performs the map first, while others wait
        # then all load the same cached result.
        tokenized_datasets = raw_datasets.map(
            tokenize_function,
            batched=True,
            remove_columns=["sentence1", "sentence2", "idx"],
        )


    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_dataloader = DataLoader(
        tokenized_datasets["train"],
        shuffle=True,
        batch_size=train_bs,
        collate_fn=data_collator,
    )
    eval_dataloader = DataLoader(
        tokenized_datasets["validation"],
        shuffle=False,
        batch_size=train_bs,
        collate_fn=data_collator,
    )

    # -----------------------------
    # 3) Model + optimizer
    # -----------------------------
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

    if use_8bit_adam:
        try:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=lr)
            if accelerator.is_main_process:
                print("Using bitsandbytes AdamW8bit optimizer.")
        except Exception as e:
            if accelerator.is_main_process:
                print(f"Could not use 8-bit AdamW. Falling back to torch AdamW. Reason: {e}")
            from torch.optim import AdamW
            optimizer = AdamW(model.parameters(), lr=lr)
    else:
        from torch.optim import AdamW
        optimizer = AdamW(model.parameters(), lr=lr)

    # -----------------------------
    # 4) Prepare for distributed training
    # -----------------------------
    # This wraps:
    # - model into DistributedDataParallel when multi-GPU
    # - dataloaders into sharded loaders (each process sees its shard)
    # - moves model/batches to correct device
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )

    # -----------------------------
    # 5) Scheduler (compute steps AFTER prepare)
    # -----------------------------
    # After prepare, train_dataloader length is per-process, which is what we want for step counting.
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    eval_steps = max(1, num_update_steps_per_epoch // 4)

    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
    
    accelerator.wait_for_everyone() # this is for pretty printing, but inefficient.
                                    # Remove it if print is not necessary
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    print(f"[rank={rank} main={is_main}] train_len={len(train_dataloader)} eval_len={len(eval_dataloader)}")
    print(f"[rank={rank} main={is_main}] eval_steps={eval_steps} total_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone() # this is for pretty printing, but inefficient.
                                    # Remove it if print is not necessary
    # -----------------------------
    # 6) Evaluation helper (distributed-safe)
    # -----------------------------
    def run_eval(model, eval_dataloader):
        model.eval()
        metric = evaluate.load("glue", "mrpc")

        # We'll compute eval_loss as a true global mean using reduce()
        total_loss = torch.tensor(0.0, device=accelerator.device)
        total_count = torch.tensor(0.0, device=accelerator.device)

        for batch in tqdm(eval_dataloader, desc="Evaluation", disable=not accelerator.is_main_process):
            with torch.no_grad():
                # accelerator.autocast uses the mixed_precision setting you chose in Accelerator(...)
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss

            bs = batch["labels"].shape[0]
            total_loss += loss.detach() * bs
            total_count += bs

            preds = torch.argmax(outputs.logits, dim=-1)

            # gather_for_metrics collects tensors from ALL processes safely (handles last-batch sizes)
            preds_g = accelerator.gather_for_metrics(preds)
            labels_g = accelerator.gather_for_metrics(batch["labels"])

            metric.add_batch(
                predictions=preds_g.detach().cpu(),
                references=labels_g.detach().cpu(),
            )

        # Reduce sums across processes => global sums
        total_loss = accelerator.reduce(total_loss, reduction="sum")
        total_count = accelerator.reduce(total_count, reduction="sum")

        # gather_for_metrics and reduce require all ranks to participate.
        # If one process is slower, the others wait at the collective call.
        
        # Rule: All processes must execute the same sequence of collectives.
        # each process iterates over its shard. If shard sizes differ, then:
            # collectives can deadlock if processes call gather/reduce a different number of times.
        # distributed samplers often ensure each rank has the same number of steps by padding/duplicating samples.
        # TODO: What happens if two shards have different numbers of batches? What is the solution?

        m = metric.compute()
        return {
            "eval_loss": (total_loss / torch.clamp(total_count, min=1.0)).item(),
            "eval_accuracy": m["accuracy"],
            "eval_f1": m["f1"],
        }

    # -----------------------------
    # 7) Training loop (Accelerate-native accumulation)
    # -----------------------------
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0

    best_eval_loss = float("inf")
    best_dir = None  # path to best checkpoint directory

    if eval_on_start:
        eval_logs = run_eval(model, eval_dataloader)
        if accelerator.is_main_process:
            accelerator.log(eval_logs, step=global_step)
            print("Eval on start:", eval_logs)

        # Save best checkpoint (main process only)
        if accelerator.is_main_process and eval_logs["eval_loss"] < best_eval_loss:
            best_eval_loss = eval_logs["eval_loss"]
            if best_dir is None:
                best_dir = tempfile.mkdtemp(prefix="best_ckpt_")
            # unwrap_model gives the real model (not DDP wrapper)
            accelerator.unwrap_model(model).save_pretrained(best_dir, safe_serialization=True)
            tokenizer.save_pretrained(best_dir)

    progress_bar = tqdm(
        total=num_training_steps, desc="Optimizer Updates",
        disable=not accelerator.is_main_process, leave = True
    )

    for epoch in range(num_train_epochs):
        model.train()

        for step, batch in enumerate(train_dataloader):

            # accelerator.accumulate(model) automatically:
            # - handles gradient accumulation
            # - only syncs gradients across GPUs on accumulation boundary
            # #### effective global batch per optimizer update: gacc_steps * n_gpu * batch_size
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                    # do NOT manually divide; accumulate() handles it correctly.
                    # See the accelerator.backward source code

                # For logging: keep the raw (undivided) loss value
                running_loss += loss.detach().float().item()
                running_mb += 1

                # Distributed-safe backward (handles scaler internally for fp16)
                accelerator.backward(loss)

                # Only do clip/step on accumulation boundary
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), max_grad_norm)

                    # This is also a collective call.
                    # If a process has reached a sync boundary and gets here,
                        # It waits for all other processes to reach this stage too.
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    
                    lr_scheduler.step()
                    scheduler_steps += 1

            # We count "global_step" in terms of optimizer updates
            if accelerator.sync_gradients:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    # local scalars
                    avg_train_loss = running_loss / max(1, running_mb)
                    # lr_val = lr_scheduler.get_last_lr()[0]
                    lr_val = optimizer.param_groups[0]["lr"]
                
                    # gather across ranks -> tensors of shape [num_processes]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    
                    print(f"[rank {accelerator.process_index}] gathering values")
                    loss_all = accelerator.gather(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather(lr_t).detach().cpu().tolist()
                    print(f"[rank {accelerator.process_index}] gathered values")
                    
                    # Only main logs
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            accelerator.log({
                                f"train_loss/rank_{r}": l,
                                f"learning_rate/rank_{r}": lr_
                            }, step=global_step)
                        print("logged to wandb")

                    running_loss = 0.0
                    running_mb = 0

                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(model, eval_dataloader)

                    if accelerator.is_main_process:
                        accelerator.log(eval_logs, step=global_step)
                        print(f"Step {global_step} eval:", eval_logs)

                        # Save new best checkpoint (overwrite same directory)
                        if eval_logs["eval_loss"] < best_eval_loss:
                            best_eval_loss = eval_logs["eval_loss"]
                            if best_dir is None:
                                best_dir = tempfile.mkdtemp(prefix="best_ckpt_")
                            accelerator.unwrap_model(model).save_pretrained(best_dir, safe_serialization=True)
                            tokenizer.save_pretrained(best_dir)

    if accelerator.is_main_process:
        print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    
    
    # -----------------------------
    # 8) Push best checkpoint to W&B as artifact (main process only)
    # -----------------------------
    accelerator.wait_for_everyone()
    progress_bar.close()

    if accelerator.is_main_process:
        assert best_dir is not None, "Best checkpoint directory was never created."

        # Access the underlying wandb run created by accelerate trackers
        wb_run = accelerator.get_tracker("wandb").run

        artifact = wandb.Artifact(
            name=f"{wandb_run_name}-best",
            type="model",
            metadata={
                "best_eval_loss": float(best_eval_loss),
                "best_step": int(global_step),
                "checkpoint": checkpoint,
                "train_bs": train_bs,
                "gacc_steps": gacc_steps,
                "effective_batch_size": train_bs * gacc_steps * accelerator.num_processes,
                "lr": lr,
                "lr_scheduler_type": lr_scheduler_type,
                "mixed_precision": mixed_precision,
                "use_8bit_adam": bool(use_8bit_adam),
            },
        )
        artifact.add_dir(best_dir)
        wb_run.log_artifact(artifact)

    # Finish trackers cleanly (important in notebooks)
    accelerator.end_training()


if __name__ == "__main__":
    main()

In [ ]:
!rm -rf wandb
!ls
# ! cat main.py | grep accelerator.gather

In [ ]:
# why the interna;
! accelerate launch --num_processes 2 main.py